In [1]:
"""
2025-11-20-Add 1mL Arresting Buffer to Rows B,C,D,F,G,H, and diluted 1:10 vertically in Deepwell plate.

Setup:
AB trough in 19[0]
DW plate in 13[0]
"""


'\n2025-11-20-Add 1mL Arresting Buffer to Rows B,C,D,F,G,H, and diluted 1:10 vertically in Deepwell plate.\n\nSetup:\nAB trough in 19[0]\nDW plate in 13[0]\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from typing import List, Iterator

from pylabrobot.liquid_handling.standard import Mix
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
# from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
# from pylabrobot.resources.opentrons.tube_racks import (
    # opentrons_24_tuberack_generic_1point5ml_snapcap_short,
# )
# from pylabrobot.resources.tube_adapter import TubeRackAdapter
from pylabrobot.resources.agenbio.plates import AGenBio_1_troughplate_100000uL_Fl
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources import (
    hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL, # 1000 µL filtered
    hamilton_96_tiprack_10uL_filter, # 10 µL filtered
)


###############################################################################
# 0) build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

 

tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)


tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")
tiprack_50 = hamilton_96_tiprack_50uL_filter("tips_01")
tiprack_10 = hamilton_96_tiprack_10uL_filter("tips_02")
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10

# Trough plate on rails=19, module slot 0
trough_module = Hamilton_MFX_plateholder_DWP_metal_tapped("trough_module")
car_19 = MFX_CAR_L5_base("car_19", modules={0:trough_module})
lh.deck.assign_child_resource(car_19, rails=19)
ab_trough = AGenBio_1_troughplate_100000uL_Fl("ab_trough")
trough_module.assign_child_resource(ab_trough)

# BioER DW plate rails=13, module slot 0
module_holding_dw_plate = Hamilton_MFX_plateholder_DWP_metal_tapped("module_holding_dw_plate")
car_13 = MFX_CAR_L5_base("car_13", modules={0:module_holding_dw_plate})
lh.deck.assign_child_resource(car_13, rails=13)
dw_plate = BioER_96_wellplate_Vb_2200uL("dw_plate")
module_holding_dw_plate.assign_child_resource(dw_plate)


2026-02-23 15:02:00,974 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-23 15:02:00,982 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-23 15:02:00,985 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-23 15:02:04,144 - pylabrobot - INFO - Running backend initialization procedure.
/tmp/ipykernel_1344052/1082131469.py:44: DeprecationWarning: Hamilton_MFX_plateholde

In [4]:
# fill rows B,C,D and F,G,H with 1 mL AB
async def fill_empty_wells(tipColumn=1):
    channel_sets = [[1, 2, 3], [5, 6, 7]]
    row_spans    = [("B", "D"), ("F", "H")]  # pass1=B:D, pass2=F:H
    
    for use_ch, (start_row, end_row) in zip(channel_sets, row_spans):
        tip_slice  = f"{start_row}{tipColumn}:{end_row}{tipColumn}"
        await lh.pick_up_tips(tiprack_1000[tip_slice], use_channels=use_ch)

        for col in range(1, 13):
            n = len(use_ch)  # 3 channels per pass
            await lh.aspirate(
                ab_trough["A1"] * 3,
                vols=[1000] * 3,
                use_channels=use_ch,
                liquid_height=[2] * 3,
            )
            dest_slice = f"{start_row}{col}:{end_row}{col}"
            await lh.dispense(
                dw_plate[dest_slice],
                vols=[1000] * 3,
                use_channels=use_ch,
                liquid_height=[20] * 3,
                blow_out=[1] * 3,
            )
            
        await lh.drop_tips(tiprack_1000[tip_slice], use_channels=use_ch) 

async def mix_and_dilute(tipColumn=1):
    assert 1 <= tipColumn <= 10, "Need three adjacent tip columns available in the rack."

    # rack two tips, channels 1,5
    # mix well A1, E1
    # transfer 100ul from A1 into well B1, E1 into F1
    # mix well B1, F1
    # transfer 100ul from B1 into well C1, F1 into G1
    # mix well C1, G1
    # transfer 100ul from C1 into well D1, G1 into H1
    # mix well D1, H1
    # discard tips, channels 1,5
    # repeat for remaining columns e.g. A2..A12
    # Serial mix-and-dilute along A→D and E→H using channels 1 and 5
    # await lh.pick_up_tips((tiprack_1000["A12"]+tiprack_1000["E12"]), use_channels=[0,4])
    # Mix
    tip_col_cycles = [tipColumn, tipColumn+1, tipColumn+2] # how many col of tips will this require? 3.
    tip_row_cycles = [["A","B"], ["C","D"], ["E","F"], ["G","H"]] # how do we group the tips? A+B, next to each other
    channel_cycles = [[0,4], [1,5], [2,6], [3,7]]
    ROW_PAIRS = [["A","B","C","D"], ["E","F","G","H"]]
    for col in range(1,13): 
        cycle_idx = (col - 1) // 4          # 0 for cols 1-4, 1 for 5-8, 2 for 9-12
        channel_idx = (col -1) % 4      # which set of channels to use
        row_idx   = (col - 1) % 4   
        tipcol    = tip_col_cycles[cycle_idx]
        r1, r2    = tip_row_cycles[row_idx]
        tip_slice = f"{r1}{tipcol}:{r2}{tipcol}"  # e.g., "A1:B1", "C1:D1", ...
        CHANNELS = [channel_cycles[channel_idx][0], channel_cycles[channel_idx][1]]

        await lh.pick_up_tips(tiprack_1000[tip_slice], use_channels=CHANNELS)
        for i in range(len(ROW_PAIRS[0]) - 1):
            currentrow = [row[i] for row in ROW_PAIRS] # ["A", "E"]
            nextrow = [row[i+1] for row in ROW_PAIRS] # ["B", "F"]
            # Mix
            await lh.aspirate(
                    dw_plate[f"{currentrow[0]}{col}"]+dw_plate[f"{currentrow[1]}{col}"], 
                    vols=[1000]*2,
                    use_channels=CHANNELS,
                    liquid_height = [20]*2,
                    auto_surface_following_distance=True,
                    flow_rates=[400]*2,
                )
            await lh.dispense(
                    dw_plate[f"{currentrow[0]}{col}"]+dw_plate[f"{currentrow[1]}{col}"],
                    vols=[1000]*2,
                    use_channels=CHANNELS,
                    liquid_height = [4]*2, 
                    flow_rates=[400]*2,
                    mix=[Mix(volume=800, repetitions=2, flow_rate=400)]*2,           
                    blow_out=[1]*2, 
                    settling_time=[1]*2,
                )
            # Dilute
            await lh.aspirate(
                    dw_plate[f"{currentrow[0]}{col}"]+dw_plate[f"{currentrow[1]}{col}"], 
                    vols=[100]*2,
                    use_channels=CHANNELS,
                    liquid_height = [10]*2,
                    auto_surface_following_distance=True,
                )
            await lh.dispense(
                    dw_plate[f"{nextrow[0]}{col}"]+dw_plate[f"{nextrow[1]}{col}"],
                    vols=[100]*2,
                    use_channels=CHANNELS,
                    liquid_height = [4]*2, 
                    mix=[Mix(volume=1000, repetitions=2, flow_rate=400)]*2,           
                    blow_out=[1]*2, 
                    settling_time=[1]*2
                )
        await lh.discard_tips()

In [5]:
BEGIN_TIP_COLUMN = 6
# TODO rack 6 tips at 1,2,3 and 5,6,7 instead of just 3 tips at 1,2,3. Decrease fill time by 1/2.
await fill_empty_wells(tipColumn=BEGIN_TIP_COLUMN) # ~13m

await mix_and_dilute(tipColumn=BEGIN_TIP_COLUMN) # ~42m


In [6]:
# 
# await lh.discard_tips()
# Dispense into B,C,D,F,G,H of this column
# await lh.dispense(
#     ab_trough["A1"],
#     vols=[1000] * 3,
#     use_channels=[1,2,3],
#     liquid_height=[4] * 3,
#     blow_out=[1] * 3,
# )
# tip_spots = [_first(tiprack_1000[f"{r}{12}"]) for r in ROWS_FOR_CHANNELS]
# await lh.drop_tips(tiprack_1000["A1:B1"], use_channels=[1,2])
# await lh.drop_tips(tiprack_1000["F12:H12"], use_channels=[5,6,7])
# await lh.drop_tips((tiprack_1000["A12"]+tiprack_1000["E12"]), use_channels=[0,4])
# await lh.discard_tips()
# await lh.dispense(
#           dw_plate["A1"]+dw_plate["E1"],
#           vols=[1000]*2,
#           use_channels=[0,4],
#           liquid_height = [20]*2, 
#           blow_out=[1]*2, 
#           settling_time=[1]*2
#       )

In [7]:
# await lh.dispense(water_plate["A1"]*8, vols=[10]*8, liquid_height=[2]*8, use_channels=CHANNELS_8)
# await lh.drop_tips(tiprack_1000["B2"], use_channels=[CHANNEL_DILUTE])
# await lh.discard_tips()
# await lh.stop()